In [1]:
!pip install metaworld@git+https://github.com/Farama-Foundation/Metaworld.git@master

  Cloning https://github.com/Farama-Foundation/Metaworld.git (to revision master) to /tmp/pip-install-bmj18ddv/metaworld_4bf363ece3c743349b124d62d305fb55
  Running command git clone --filter=blob:none --quiet https://github.com/Farama-Foundation/Metaworld.git /tmp/pip-install-bmj18ddv/metaworld_4bf363ece3c743349b124d62d305fb55
  Resolved https://github.com/Farama-Foundation/Metaworld.git to commit 918cd0d3abb6c45f213d72730a06ef119d84218e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 27.9 MB/s eta 0:00:00
  Created wheel for metaworld: filename=metaworld-3.0.0-py3-none-any.whl size=36660399 sha256=d960fb28bdab57c578d10f2de94fae87c11dbcfcedbe96f6245fd2d105ef1330
  Stored in directory: /tm

In [7]:
import metaworld
import random

print("Loading Meta-World ML1 Benchmark...")
# Load the ML1 benchmark (contains the basic reach task)
ml1 = metaworld.ML1('reach-v3')

# Initialize the environment
env = ml1.train_classes['reach-v3']()

# Assign a random task (goal position) to the environment
task = random.choice(ml1.train_tasks)
env.set_task(task)

# Reset the environment to get the initial state
state, info = env.reset()

print("\n--- Installation Successful ---")
print(f"Observation Space Shape: {state.shape}") # Should be (39,)
print(f"Action Space Shape: {env.action_space.shape}") # Should be (4,)

# Take a random action to test the physics engine
action = env.action_space.sample()
next_state, reward, done, truncated, info = env.step(action)

print(f"Step completed. Reward: {reward:.4f}")

Loading Meta-World ML1 Benchmark...

--- Installation Successful ---
Observation Space Shape: (39,)
Action Space Shape: (4,)
Step completed. Reward: 1.3649


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import collections


from sklearn.decomposition import PCA
from torch.distributions import Normal
from sklearn.preprocessing import StandardScaler
class RewardScaler:
    def __init__(self, gamma=0.99):
        self.gamma = gamma
        self.running_return = 0.0
        self.variance = 1e-4

    def scale(self, reward):
        self.running_return = self.running_return * self.gamma + reward
        # Update rolling variance
        self.variance = 0.99 * self.variance + 0.01 * (self.running_return ** 2)
        # Scale the reward down by the standard deviation of the returns
        return reward / (np.sqrt(self.variance) + 1e-8)

    def reset(self):
        self.running_return = 0.0
class MetaWorldHybridCompressor:
    def __init__(self, n_components=8, buffer_size=1000):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components)
        self.is_fitted = False
        self.warmup_buffer = []
        self.buffer_size = buffer_size

    def warmup(self, state_39d):
        if not self.is_fitted:
            self.warmup_buffer.append(state_39d)
            if len(self.warmup_buffer) >= self.buffer_size:
                # 1. Convert buffer to numpy array
                data = np.vstack(self.warmup_buffer)

                # 2. Fit the Scaler and transform the data
                scaled_data = self.scaler.fit_transform(data)

                # 3. Fit the PCA on the scaled data
                self.pca.fit(scaled_data)
                self.is_fitted = True

                variance = np.sum(self.pca.explained_variance_ratio_)
                print(f"[DQACS] Scaler & PCA fitted. Retained Variance: {variance:.4f}")

                if variance < 0.85:
                    print(f"WARNING: Variance is low. Consider increasing n_components from {self.pca.n_components}.")

        return self.is_fitted

    def compress(self, state_39d):
        if not self.is_fitted:
            raise RuntimeError("Must call warmup() before training.")

        if state_39d.ndim == 1:
            state_39d = state_39d.reshape(1, -1)

        # 1. Scale the incoming state
        scaled_state = self.scaler.transform(state_39d)

        # 2. Compress the scaled state
        state_8d = self.pca.transform(scaled_state)

        # 3. Normalize for Quantum Angle Encoding [-pi, pi]
        state_8d_normalized = np.tanh(state_8d) * np.pi
        return torch.tensor(state_8d_normalized, dtype=torch.float32)

class HybridContinuousAgent(nn.Module):
    def __init__(self, q_dim=8, action_dim=4):
        super().__init__()
        # MOCK VQC for testing classical tensor flow
        self.q_layer = nn.Sequential(nn.Linear(q_dim, 16), nn.Tanh(), nn.Linear(16, q_dim))

        self.actor_mean = nn.Linear(q_dim, action_dim)
        self.actor_logstd = nn.Parameter(torch.zeros(1, action_dim))
        self.critic_head = nn.Linear(q_dim, 1)

    def get_action_and_value(self, state_8d, action=None):
        q_out = self.q_layer(state_8d)
        action_mean = self.actor_mean(q_out)
        action_logstd = self.actor_logstd.expand_as(action_mean)
        action_std = torch.exp(action_logstd)

        dist = Normal(action_mean, action_std)
        if action is None:
            action = dist.sample()

        # Sum the log probs across the 4 dimensions
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        value = self.critic_head(q_out)
        return action, log_prob, entropy, value

# ---------------------------------------------------------
# Environment Setup (Toggle Real vs Dummy)
# ---------------------------------------------------------

import metaworld
import random
ml1 = metaworld.ML1('reach-v3') # Simple 39D/4D task to test
env = ml1.train_classes['reach-v3']()
task = random.choice(ml1.train_tasks)
env.set_task(task)


# ---------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------
NUM_STEPS = 2048         # Steps per rollout
BATCH_SIZE = 64
EPOCHS = 4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_COEF = 0.2
ENT_COEF = 0.01
LR = 3e-4

def train():
    compressor = MetaWorldHybridCompressor(n_components=8, buffer_size=1000)
    agent = HybridContinuousAgent(q_dim=8, action_dim=4)
    optimizer = optim.Adam(agent.parameters(), lr=LR, eps=1e-5)

    print("--- 1. Generating PCA Warmup Buffer ---")
    state = env.reset()
    if type(state) is tuple: state = state[0]

    is_ready = False
    while not is_ready:
        action = np.random.uniform(-1, 1, size=(4,))

        # EXPLICITLY UNPACK 5 VALUES
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated # Reset if either is true

        is_ready = compressor.warmup(next_state)

        if done:
            state = env.reset()
            if type(state) is tuple: state = state[0]
        else:
            state = next_state
    reward_scaler = RewardScaler(gamma=GAMMA)
    print("\n--- 2. Starting PPO Training Loop ---")

    for update in range(1, 11): # Test for 10 updates
        # Rollout Buffers
        b_states, b_actions, b_logprobs, b_rewards, b_values, b_dones = [], [], [], [], [], []

        state = env.reset()
        if type(state) is tuple: state = state[0]

        ep_rewards = []

        # --- Collection Phase ---
        for step in range(NUM_STEPS):
            state_8d = compressor.compress(state)

            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(state_8d)

            action_np = action.cpu().numpy()[0]

            # EXPLICITLY UNPACK 5 VALUES
            next_state, reward, terminated, truncated, info = env.step(action_np)
            done = terminated or truncated # Reset if either is true

            # Scale the reward
            scaled_reward = reward_scaler.scale(reward)


            b_states.append(state_8d)
            b_actions.append(action)
            b_logprobs.append(logprob)
            b_rewards.append(scaled_reward)
            b_values.append(value)
            b_dones.append(done)

            if done:
                state = env.reset()
                if type(state) is tuple: state = state[0]
                reward_scaler.reset()
            else:
                state = next_state

            ep_rewards.append(reward)

        # Convert buffers to tensors
        b_states = torch.cat(b_states)
        b_actions = torch.cat(b_actions)
        b_logprobs = torch.cat(b_logprobs)
        b_rewards = torch.tensor(b_rewards, dtype=torch.float32)
        b_values = torch.cat(b_values).squeeze()
        b_dones = torch.tensor(b_dones, dtype=torch.float32)

        # --- Generalized Advantage Estimation (GAE) ---
        with torch.no_grad():
            next_state_8d = compressor.compress(state)
            _, _, _, next_value = agent.get_action_and_value(next_state_8d)
            next_value = next_value.squeeze()

            advantages = torch.zeros_like(b_rewards)
            lastgaelam = 0
            for t in reversed(range(NUM_STEPS)):
                nextnonterminal = 1.0 - b_dones[t]
                nextvalues = next_value if t == NUM_STEPS - 1 else b_values[t + 1]
                delta = b_rewards[t] + GAMMA * nextvalues * nextnonterminal - b_values[t]
                advantages[t] = lastgaelam = delta + GAMMA * GAE_LAMBDA * nextnonterminal * lastgaelam
            returns = advantages + b_values

        # --- Optimization Phase ---
        b_inds = np.arange(NUM_STEPS)
        for epoch in range(EPOCHS):
            np.random.shuffle(b_inds)
            for start in range(0, NUM_STEPS, BATCH_SIZE):
                end = start + BATCH_SIZE
                mb_inds = b_inds[start:end]

                _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_states[mb_inds], b_actions[mb_inds])
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()

                # Policy Loss (Clipped Surrogate)
                mb_advantages = advantages[mb_inds]
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - CLIP_COEF, 1 + CLIP_COEF)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                # Value Loss
                v_loss = 0.5 * ((newvalue.squeeze() - returns[mb_inds]) ** 2).mean()

                # Total Loss
                entropy_loss = entropy.mean()
                loss = pg_loss - ENT_COEF * entropy_loss + v_loss

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
                optimizer.step()

        print(f"Update: {update:02d} | Avg Rollout Reward: {np.mean(ep_rewards):.4f} | Policy Loss: {pg_loss.item():.4f} | Value Loss: {v_loss.item():.4f}")

if __name__ == "__main__":
    train()

--- 1. Generating PCA Warmup Buffer ---
[DQACS] Scaler & PCA fitted. Retained Variance: 0.9644

--- 2. Starting PPO Training Loop ---
Update: 01 | Avg Rollout Reward: 0.6424 | Policy Loss: 0.0088 | Value Loss: 0.0910
Update: 02 | Avg Rollout Reward: 0.6568 | Policy Loss: 0.0022 | Value Loss: 0.0064
Update: 03 | Avg Rollout Reward: 0.6877 | Policy Loss: -0.0012 | Value Loss: 0.0125
Update: 04 | Avg Rollout Reward: 0.7418 | Policy Loss: -0.0273 | Value Loss: 0.0064
Update: 05 | Avg Rollout Reward: 0.7397 | Policy Loss: -0.0085 | Value Loss: 0.0079
Update: 06 | Avg Rollout Reward: 0.7875 | Policy Loss: -0.0105 | Value Loss: 0.0080
Update: 07 | Avg Rollout Reward: 1.2205 | Policy Loss: -0.0052 | Value Loss: 0.0076
Update: 08 | Avg Rollout Reward: 1.1190 | Policy Loss: -0.0210 | Value Loss: 0.0128
Update: 09 | Avg Rollout Reward: 1.4846 | Policy Loss: -0.0275 | Value Loss: 0.0095
Update: 10 | Avg Rollout Reward: 1.2743 | Policy Loss: -0.0057 | Value Loss: 0.0108
